# Tutorial de Owlready2 con `GameOntologyv3.owl`

**Objetivo.** Aprender a trabajar con una ontología OWL desde Python usando **Owlready2**:

1. Cargar la ontología y detectar errores de carga.
2. Explorar clases, propiedades e individuos.
3. Consultar jerarquías, dominios, rangos e inversas.
4. Crear individuos y afirmaciones.
5. Ampliar la ontología con **clases y propiedades nuevas**.
6. Añadir características a los roles: funcional, inversamente funcional, simétrica y transitiva.
7. Ejecutar un razonador básico.

La ontología de partida es `GameOntologyv3.owl` (*The Video Game Ontology*). Entre sus clases aparecen `Game`, `Player`, `Character`, `Achievement`, `Item`, `Genre`, `Session`, etc., y contiene propiedades como `hasCharacter`, `hasAchievement`, `involvesPlayer`, `playsGame` e `isFriendWithPlayer`.

> **Importante:** coloca `GameOntologyv3.owl` en la misma carpeta que este notebook.


## 1. Instalación e importación

In [1]:
# Ejecutar una sola vez si Owlready2 no está instalado
%pip install owlready2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from owlready2 import *
#from pathlib import Path

#Para comprobar que se tiene Java instalado
import subprocess
subprocess.run(["java", "-version"])

CompletedProcess(args=['java', '-version'], returncode=0)

## 2. Cargar la ontología

`get_ontology()` obtiene o crea una referencia a una ontología. La lectura real del fichero se realiza con `load()`.

Por eso podemos utilizar `try/except` para comprobar si la carga se ha realizado correctamente.


In [3]:
onto = get_ontology("GameOntologyv3.owl")

try:
    onto.load()
    print("Ontología cargada correctamente")
except FileNotFoundError:
    print("ERROR: no se encuentra el fichero", archivo)
except Exception as e:
    print("ERROR al cargar la ontología:")
    print(type(e).__name__, "-", e)

Ontología cargada correctamente


### Comprobación básica

Una segunda comprobación consiste en inspeccionar el contenido cargado.
Para acceder a las clases, instancias, propiedades, anotaciones o cualquier elemento de la ontología, se emplea la notación con 'punto'. También se admiten los corchetes []

Tenemos disponibles los siguientes métodos para acceder al contenido de la ontología. La mayoría proporciona un generador (ver la documentación de owlready para la lista completa). Estos generadores permiten iterar sobre los elementos sin la necesidad de crear una lista. Si necesitas la lista, siempre puedes generarla con `list()`

- `.classes()` acceso a las clases
- `.individuals()` acceso a las instancias
- `.object_properties()` acceso a las `ObjectProperties` (relaciones entre clases)
- `.data_properties()`  acceso a las `DataProperties` (atributos)
- `.properties()` acceso a todas las propiedades definidas en la ontología

La ontología original contiene clases y propiedades, pero no define individuos propios, por lo que inicialmente `onto.individuals()` puede estar vacío.


In [4]:
print("Número de clases:", len(list(onto.classes())))
print("Número de propiedades de objeto:", len(list(onto.object_properties())))
print("Número de propiedades de datos:", len(list(onto.data_properties())))
print("Número de individuos:", len(list(onto.individuals())))

Número de clases: 37
Número de propiedades de objeto: 33
Número de propiedades de datos: 6
Número de individuos: 0


## 3. Explorar las clases

Owlready2 proporciona métodos para recorrer la jerarquía de clases:
- el **atributo** `ìs_a` hace referencia a la superclase
- el **método** `subclasses()` devuelve un generador de las subclases **directas**
- los métodos `.ancestors()` y `.descendants()` devuelven el conjunto (set) completo de clases ascendentes y descendentes, incluyendo la propia clase

In [5]:
for clase in onto.classes():
    print(clase.name)

Thing
Game
Agent
Achievement
Character
Feature
Item
Genre
Leaderboard
PlayingArea
Session
Player
InstantaneousEvent
PopulatedPlace
GameProduct
InAppPurchaseEvent
ProductOrService
Collection
Completion
Curiosity
Fandom
GainEvent
GameEvent
Product
HardMode
LoseEvent
Loyalty
Luck
MenuEvent
Minigame
Multiplayer
Paragon
PlayerEvent
SpecialPlayStyle
Tutorial
Veteran
Virtuosity


Podemos acceder directamente a una clase por su nombre. Por ejemplo, `Game`, `Player` y `Character` existen en la ontología.

In [6]:
print(onto.Game)
print(onto.Player)
print(onto.Character)

print("\nSuperclases de Character:")
print(onto.Character.is_a)

print("\nSubclases directas de Achievement:")
for c in onto.Achievement.subclasses():
    print("-", c.name)

GameOntologyv3.Game
GameOntologyv3.Player
GameOntologyv3.Character

Superclases de Character:
[owl.Thing]

Subclases directas de Achievement:
- Collection
- Completion
- Curiosity
- Fandom
- HardMode
- Loyalty
- Luck
- Minigame
- Multiplayer
- Paragon
- SpecialPlayStyle
- Tutorial
- Veteran
- Virtuosity


## 4. Explorar propiedades, dominio, rango e inversas

In [7]:
for prop in onto.object_properties():
    print(prop.name)

creator
hasAchievement
hasCharacter
hasFeature
hasGameGenre
hasItem
hasLeaderboard
isLeaderboardInGame
hasPlayingArea
involvesAchievement
involvesCharacter
involvesPlayer
isAchievedInSession
isAchievementInGame
isCharacterInGame
isCharacterInSession
isEventAssociatedToPlayer
isEventInGame
isEventInSession
isEventRelatedToItem
isEventTriggeredByCharacter
isFriendWithPlayer
knows
isItemInGame
isPlayerInSession
isSessionInGame
livesIn
ownsAchievement
ownsCharacter
ownsItem
playsGame
purchasesGameOffering
unlocksAchievement


In [8]:
p = onto.hasCharacter

print("Propiedad:", p)
print("Dominio:", p.domain)
print("Rango:", p.range)
print("Propiedad inversa:", p.inverse_property)

Propiedad: GameOntologyv3.hasCharacter
Dominio: [GameOntologyv3.Game]
Rango: [GameOntologyv3.Character]
Propiedad inversa: GameOntologyv3.isCharacterInGame


En la ontología, `hasCharacter` relaciona juegos con personajes. Si además está definida su propiedad inversa, Owlready2 permite navegar en ambos sentidos.

También podemos inspeccionar propiedades de datos como `username`.

In [9]:
print("Dominio de username:", onto.username.domain)
print("Rango de username:", onto.username.range)

Dominio de username: [GameOntologyv3.Player]
Rango de username: [<class 'str'>]


## 5. Crear individuos

Los individuos (instancias) se crean como cualquier otra instancia en Python, ya que las clases de la ontología son clases Python. Algunos parámetros adicionales relacionados con OWL están disponibles (ver la documentación de Owlready2 para más información). Los individuos están disponibles inmediatamente dentro de la ontología, no es necesario recargarla con `.load()`

Importante: el nombre de la instancia es externo en Python. Dentro de la ontología, se emplea el nombre que se le pasa como parámetro al crearla.

El método `.instances()` devuelve un generador para iterar sobre todas las instancias de una clase dada (el método `.individuals()` devuelve todas las instancias de la ontología)

Vamos a crear algunos individuos para construir una pequeña ABox sobre la ontología existente.


In [10]:
juego1 = onto.Game("juego1")
juego2 = onto.Game("juego2")

ana = onto.Player("ana")
luis = onto.Player("luis")
marta = onto.Player("marta")

heroina = onto.Character("heroina")
villano = onto.Character("villano")

print(juego1, ana, heroina)

print("Jugadores:")
for entity in onto.Player.instances():
    print(entity)

GameOntologyv3.juego1 GameOntologyv3.ana GameOntologyv3.heroina
Jugadores:
GameOntologyv3.ana
GameOntologyv3.luis
GameOntologyv3.marta


### Añadir relaciones y datos

En Owlready2, una propiedad se comporta como multivaluada por defecto, salvo que esté declarada como *funcional*. Las propiedades multivaluadas se manejan como listas.


In [11]:
juego1.hasCharacter.append(heroina)
juego1.hasCharacter.append(villano)

ana.playsGame.append(juego1)
luis.playsGame.append(juego1)

ana.username = ["AnaPlayer"]

print("Personajes de juego1:", juego1.hasCharacter)
print("Juegos de Ana:", ana.playsGame)
print("Username:", ana.username)

Personajes de juego1: [GameOntologyv3.heroina, GameOntologyv3.villano]
Juegos de Ana: [GameOntologyv3.juego1]
Username: ['AnaPlayer']


## 6. Ampliar la ontología con clases nuevas

Podemos ampliar una ontología creando nuevas clases a partir de las clases existentes.
Para crear una clase raíz hay que heredar de la clase principal `owlready2.Thing`. Se especifica la ontología en la que se crea esta clase con `with`.
Las subclases se definen heredando de las clases ya generadas. Se admite herencia múltiple.

En este ejemplo se crea la clase `Heroe` como subclase de `Character`:

**Heroe ⊑ Character**

~~~python
with onto:
    class Heroe(onto.Character):
        pass
~~~

- `with onto:` indica que la nueva clase se crea dentro de la ontología `onto`.
- `class Heroe(...)` crea la nueva clase `Heroe`.
- `onto.Character` indica que `Heroe` es una subclase de la clase `Character` existente en la ontología.
- `pass` indica que, por el momento, no añadimos ninguna otra definición a la clase.

A partir de esta definición, todo individuo que pertenezca a `Heroe` será también un `Character`.

In [12]:
with onto:
    class Heroe(onto.Character):
        pass

# Creamos un individuo de la clase para comprobar que se ha creado correctamente
mario = onto.Heroe("mario")
print("Mario pertenece a la clase:", mario.is_a)

Mario pertenece a la clase: [GameOntologyv3.Heroe]


Ahora añadimos clases a partir de equivalencias.

### `JugadorActivo`

Queremos definir como `JugadorActivo` a cualquier `Player` que juegue al menos a algún `Game`.

En DL:

**JugadorActivo ≡ Player ⊓ ∃ playsGame.Game**

La equivalencia es importante: no solo establece condiciones necesarias, sino también suficientes para que el razonador pueda clasificar individuos.

In [13]:
with onto:
    class JugadorActivo(onto.Player):
        equivalent_to = [
            onto.Player & onto.playsGame.some(onto.Game)
        ]

donde:

- `equivalent_to` establece que la clase que estamos definiendo es **equivalente** (≡) a la expresión que aparece entre corchetes.
- `onto.Player` representa la clase `Player`.
- `&` representa la **intersección** (⊓).
- `playsGame.some(Game)` representa la **restricción existencial** (∃): juega al menos a un `Game`.

### Dos clases disjuntas

Creamos `JugadorCasual` y `JugadorCompetitivo` y declaramos que son disjuntas.

En DL, el siguiente código corresponde a:

**JugadorCasual ⊑ Player**

**JugadorCompetitivo ⊑ Player**

**JugadorCasual ⊓ JugadorCompetitivo ⊑ ⊥**

Por tanto:

- Todo `JugadorCasual` es un `Player`.
- Todo `JugadorCompetitivo` es un `Player`.
- `JugadorCasual` y `JugadorCompetitivo` son **clases disjuntas**: ningún individuo puede pertenecer simultáneamente a ambas clases.

In [14]:
with onto:
    class JugadorCasual(onto.Player):
        pass

    class JugadorCompetitivo(onto.Player):
        pass

    AllDisjoint([JugadorCasual, JugadorCompetitivo])

## 7. Crear propiedades con diferentes características

Las propiedades son clases heredadas de `ObjectProperty` o de `DataProperty`. El origen y el destino de las propiedades se establecen a través de los atributos `domain` y `range`. Se puede usar la notación `dominio >> rango` para abreviar. Pueden ser listas, ya que OWL permite que sea la intersección de varios dominios o rangos en una propiedad. En el caso de propiedades `DataProperty`, Owlready2 soporta los siguientes tipos de datos
- int
- float
- bool
- str (string)
- datetime.date
- datetime.time
- datetime.datetime

Además, se puede indicar que la propiedad es
- funcional `FunctionalProperty`
- transitiva `TransitiveProperty`
- simétrica `SymmetricProperty`
- asimétrica `AsymmetricProperty`
- reflexiva `ReflexiveProperty`
- irreflexiva `IrreflexiveProperty`

Esta parte permite relacionar directamente Owlready2 con las características de roles estudiadas en DL.
Cada característica se indica mediante una clase de Python que hereda de `ObjectProperty` y de la clase correspondiente a la característica.

### 7.1. Propiedad funcional

Un juego tiene como máximo un personaje principal.

**hasMainCharacter(Game, Character)**

Si la propiedad es funcional, un mismo `Game` no puede tener dos valores diferentes para ella. Veremos más adelante cómo detectar una posible inconsistencia si se intenta asignar dos personajes principales distintos a un mismo juego.


In [15]:
with onto:
    class hasMainCharacter(ObjectProperty, FunctionalProperty):
        domain = [onto.Game]
        range = [onto.Character]

juego1.hasMainCharacter = heroina
print(juego1.hasMainCharacter)

GameOntologyv3.heroina


### 7.2. Propiedad inversamente funcional

Creamos un identificador como **individuo**, para poder utilizar una Object Property inversamente funcional.

Si dos jugadores tienen el mismo identificador, el razonador puede concluir que representan al mismo individuo. Lo veremos más adelante.


In [16]:
with onto:
    class PlayerIdentifier(Thing):
        pass

    class hasPlayerIdentifier(ObjectProperty, InverseFunctionalProperty):
        domain = [onto.Player]
        range = [PlayerIdentifier]

id_001 = PlayerIdentifier("id_001")
ana.hasPlayerIdentifier.append(id_001)

### 7.3. Propiedad simétrica

Si A es compañero de equipo de B, entonces B es compañero de equipo de A.
Solamente indicando que Ana es compañera de equipo de Luis, el razonador puede inferir automáticamente que Luis es compañero de equipo de Ana.


In [17]:
with onto:
    class isTeammateOf(ObjectProperty, SymmetricProperty):
        domain = [onto.Player]
        range = [onto.Player]

if luis not in ana.isTeammateOf:  #Para no añadirlo varias veces
    ana.isTeammateOf.append(luis)

print("Ana →", ana.isTeammateOf)
print("Luis →", luis.isTeammateOf)

Ana → [GameOntologyv3.luis]
Luis → [GameOntologyv3.ana]


### 7.4. Propiedad transitiva

Si un jugador está por encima de otro en una jerarquía y este está por encima de un tercero, podemos hacer transitiva la relación `outranks`. Veremos más adelante cómo el razonador puede inferir que Ana está por encima de Marta.


In [18]:
with onto:
    class outranks(ObjectProperty, TransitiveProperty):
        domain = [onto.Player]
        range = [onto.Player]

if luis not in ana.outranks:
    ana.outranks.append(luis)
if marta not in luis.outranks:
    luis.outranks.append(marta)

print("Ana →", ana.outranks)
print("Luis →", luis.outranks)
print("Marta →", marta.outranks)

Ana → [GameOntologyv3.luis]
Luis → [GameOntologyv3.marta]
Marta → []


## 8.Razonamiento básico

#### Obtención de clases indirectas

`is_a` permite consultar las clases a las que un individuo pertenece directamente:

~~~python
print(mario.is_a)
~~~

`INDIRECT_is_a` incluye además las clases obtenidas indirectamente a través de la jerarquía de clases:

~~~python
print(mario.INDIRECT_is_a)
~~~

Por ejemplo, si:

**Heroe ⊑ Character** y **Heroe(mario)**

entonces `mario.is_a` contiene `Heroe`, mientras que `mario.INDIRECT_is_a` permite observar también que `mario` pertenece a `Character` y `Thing` por herencia.

In [19]:
print("Mario pertenece a la clase:", mario.is_a)
print("Mario también pertenece a las clases:", mario.INDIRECT_is_a)

Mario pertenece a la clase: [GameOntologyv3.Heroe]
Mario también pertenece a las clases: [GameOntologyv3.Character, GameOntologyv3.Heroe, owl.Thing]


En general, a diferentes métodos, se le puede anteponer «INDIRECT_» para obtener todas las entidades indirectamente relacionadas. Tiene en cuenta:
- propiedades transitivas, simétricas y reflexivas,
- herencia de propiedades (es decir, subpropiedades),
- las clases de un individuo (es decir, los valores afirmados a nivel de clase),
- herencia de clases (es decir, clases padre).
- equivalencias (clases equivalentes, individuos idénticos, etc.)



In [20]:
print("Ana (directo) →", ana.outranks)
print("Ana (indirecto) →", ana.INDIRECT_outranks)
print("Luis (directo) →", luis.outranks)
print("Luis (indirecto) →", luis.INDIRECT_outranks)
print("Marta (directo) →", marta.outranks)
print("Marta (indirecto) →", marta.INDIRECT_outranks)

Ana (directo) → [GameOntologyv3.luis]
Ana (indirecto) → [GameOntologyv3.luis, GameOntologyv3.marta]
Luis (directo) → [GameOntologyv3.marta]
Luis (indirecto) → [GameOntologyv3.marta]
Marta (directo) → []
Marta (indirecto) → []


## 9. Guardar la ontología ampliada

Las clases, propiedades e individuos que hemos creado pueden guardarse en un nuevo fichero OWL.


In [21]:
onto.save(file="GameOntology_ampliada.owl", format="rdfxml")
print("Guardada como GameOntology_ampliada.owl")

Guardada como GameOntology_ampliada.owl


# Ejercicios propuestos

### Ejercicio 1
Muestra todas las clases existentes en la ontología.

### Ejercicio 2
Muestra todas las subclases directas de `Achievement`.

A continuación, selecciona una de ellas y muestra todas sus superclases.

### Ejercicio 3
Consulta el dominio y rango de `involvesPlayer`.

### Ejercicio 4

Crea las siguientes clases:

- `JuegoDeportivo` como subclase de `Game`.
- `JuegoFutbol` como subclase de `JuegoDeportivo`.

Crea un individuo `juegoFutbol1` de la clase `JuegoFutbol`.

Consulta sus tipos directos e indirectos y comprueba la herencia producida por la jerarquía:

**JuegoFutbol ⊑ JuegoDeportivo ⊑ Game**

### Ejercicio 5

Crea una propiedad `hasEnemy` que relacione individuos de la clase `Character` con otros individuos de la clase `Character`.

Establece:

- `Character` como dominio.
- `Character` como rango.

Crea dos personajes, `mario` y `bowser`, y establece que Bowser es enemigo de Mario.

Consulta posteriormente los valores de `hasEnemy` para Mario.

### Ejercicio 6

Crea dos propiedades:

- `hasCaptain`, que relaciona `Game` con `Player`.
- `isCaptainOf`, que relaciona `Player` con `Game`.

Declara ambas propiedades como inversas.

Crea un juego y un jugador y establece únicamente:

**hasCaptain(juego1, jugador1)**

Consulta después `isCaptainOf` desde `jugador1` y comprueba qué información proporciona Owlready2.

### Ejercicio 7

Crea una propiedad `isRivalOf` entre jugadores y declárala como simétrica.

Crea dos jugadores:

`jugador1` y `jugador2`

Añade únicamente la relación:

**isRivalOf(jugador1, jugador2)**

Consulta posteriormente la relación desde ambos individuos y observa el efecto de haber declarado la propiedad como simétrica.

### Ejercicio 8

Crea una propiedad `isInside` que relacione elementos del juego y declárala como transitiva.

Crea tres individuos:

- `espada`
- `cofre`
- `habitacion`

Establece únicamente las siguientes relaciones:

**isInside(espada, cofre)**

**isInside(cofre, habitacion)**

Consulta primero los valores directos de `isInside` para `espada`.

A continuación, utiliza `INDIRECT_isInside` para consultar también las relaciones obtenidas indirectamente.

¿Podemos concluir que la espada está dentro de la habitación? ¿Por qué?

# Resumen

En este notebook hemos recorrido tres niveles:

**1. Ontología original**  
Cargar → explorar clases → propiedades → dominio/rango → jerarquías.

**2. ABox**  
Crear individuos → añadir propiedades → introducir hechos.

**3. TBox + razonamiento**  
Crear nuevas clases → definirlas mediante constructores → añadir características de roles → obtener inferencias con INDIRECT.

## Equivalencia entre Lógica Descriptiva (DL) y Owlready2

| Concepto | Lógica Descriptiva (DL)     | Owlready2 |
|---|-----------------------------|---|
| **Clase** | `Persona`                   | `class Persona(Thing): ...` |
| **Subclase** | `Jugador ⊑ Persona`         | `class Jugador(Persona): ...` |
| **Pertenencia a una clase** | `Persona(ana)`              | `ana = Persona("ana")` |
| **Intersección** | `Persona ⊓ Jugador`         | `Persona & Jugador` |
| **Unión** | `Jugador ⊔ Entrenador`      | `Jugador \| Entrenador` |
| **Complemento / negación** | `¬Jugador`                  | `Not(Jugador)` |
| **Equivalencia de clases** | `A ≡ B`                     | `A.equivalent_to = [B]` |
| **Clases disjuntas** | `A ⊓ B ⊑ ⊥`                 | `AllDisjoint([A, B])` |
| **Rol / propiedad de objeto** | `juegaA(x,y)`               | `class juegaA(ObjectProperty): ...` |
| **Relación entre individuos** | `juegaA(ana,juego1)`        | `ana.juegaA.append(juego1)` |
| **Restricción existencial** | `∃ juegaA.Game`             | `juegaA.some(Game)` |
| **Restricción universal** | `∀ juegaA.Game`             | `juegaA.only(Game)` |
| **Dominio** | Player es dominio de juegaA | `domain = [Player]` |
| **Rango** | Game es rango de juegaA     | `range = [Game]` |
| **Propiedad inversa** | `R⁻`                        | `inverse_property = ...` |
| **Propiedad funcional** | `Functional(R)`             | `FunctionalProperty` |
| **Propiedad inversamente funcional** | `InverseFunctional(R)`      | `InverseFunctionalProperty` |
| **Propiedad simétrica** | `Symmetric(R)`              | `SymmetricProperty` |
| **Propiedad transitiva** | `Transitive(R)`             | `TransitiveProperty` |
| **Propiedad reflexiva** | `Reflexive(R)`              | `ReflexiveProperty` |
| **Propiedad irreflexiva** | `Irreflexive(R)`            | `IrreflexiveProperty` |
| **Propiedad asimétrica** | `Asymmetric(R)`             | `AsymmetricProperty` |
